## Process - phase 1, before qf-batch
- Load the MSA file
- Clean data and first mapping for bdd injection
- Write the qf-batch input, one allocataire per household
- Drop duplicates
- Add default column values
- Output the cleaned beneficiary rows to parquet, for `clean_msa_2_after_qf_batch.ipynb`

## Input format
MSA delivers a `;`-separated CSV, `utf-8-sig`, whose 29 columns are described by
`en_tête_colonne_PassSport.csv`. New in 2026: the allocataire pivot identity (nom de
naissance, prénom usuel, date/commune/pays de naissance) that the quotient_familial call
needs - which is why MSA now routes through qf-batch like the CNAF.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# partners_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# partners_lib itself sits one level up, in partners/, next to the other partner folders.
partners_root = str(Path.cwd().parent)
if partners_root not in sys.path:
    sys.path.append(partners_root)

# All the DataFrame processing lives in two modules so it can be unit tested: what every
# partner routed through qf-batch shares in ../partners_lib.py, and what is specific to
# the MSA file (its column names, its date/genre encodings) in clean_msa_lib.py.
import partners_lib as partners
import clean_msa_lib as msa

load_dotenv()

msa_input_filepath = os.environ['MSA_PATHFILE_2026']
qf_batch_input_filepath = os.environ['MSA_QF_BATCH_INPUT_PATHFILE_2026']

# INSEE COG countries & territories, used to turn the birth country label into a COG code.
# Defaults to the copy shipped in partners/, shared with the other partners.
cog_pays_input_filepath = os.environ.get(
    'COG_PAYS_PATHFILE_2026', str(Path.cwd().parent / 'v_pays_territoire_2026.csv'))

# Handoff to clean_msa_2_after_qf_batch.ipynb: parquet rather than CSV so date_naissance
# stays a datetime and the NaN/'' distinction survives the round trip.
# qf-batch-workdir sits in the partners/ folder rather than this one: every partner routed
# through qf-batch shares it, so the files inside stay partner-prefixed.
msa_intermediate_filepath = os.environ.get(
    'MSA_INTERMEDIATE_PATHFILE_2026',
    str(Path.cwd().parent / 'qf-batch-workdir' / 'msa_2026_pre_qf_batch.parquet'))

In [ ]:
# Everything as str: matricules, INSEE codes and dates are identifiers, not numbers, and
# keep_default_na=False keeps an empty cell an empty string rather than a NaN.
msa_df = pd.read_csv(msa_input_filepath, sep=';', dtype='str', keep_default_na=False,
                     encoding='utf-8-sig')

print(f"{len(msa_df)} total number of rows")

In [ ]:
# The MSA export is space-padded on every column
msa_df = partners.strip_all_string_columns(msa_df)

# map MSA columns to the PSP schema (see msa.MSA_COLUMN_MAPPING)
df_psp_mapped_msa = msa.map_msa_columns(msa_df)

In [ ]:
# MSA-specific encodings: no allocataire genre column (read off the civility), 1/2 for the
# beneficiary genre, %Y%m%d dates.
df_psp_mapped_msa = msa.derive_allocataire_genre(df_psp_mapped_msa)
df_psp_mapped_msa = msa.normalize_beneficiary_genre(df_psp_mapped_msa)
df_psp_mapped_msa = msa.normalize_allocataire_birthdate(df_psp_mapped_msa)

# Birth country/place: MSA leaves the label empty for a birth in France (ISO '0'), and its
# export drops the leading zero of INSEE codes. Both are fixed on the main frame so the
# allocataire JSON carries the same values the qf-batch pivot is built from.
df_psp_mapped_msa = msa.fill_france_birth_country(df_psp_mapped_msa)
df_psp_mapped_msa = msa.pad_birthplace_insee(df_psp_mapped_msa)

In [ ]:
# Allocataire missing phone number
df_psp_mapped_msa = partners.clear_placeholder_phone_numbers(df_psp_mapped_msa)

# Allocataire's qualite
df_psp_mapped_msa = partners.normalize_allocataire_qualite(df_psp_mapped_msa)

# Allocataire's street address, joined from the 4 columns MSA splits it across, and the
# postal addressee built from the destinataire columns
df_psp_mapped_msa = msa.build_allocataire_address_fields(df_psp_mapped_msa)
df_psp_mapped_msa = msa.build_nom_adresse_postale(df_psp_mapped_msa)

# Organism & situation - MSA flags the category itself through its prestation column
df_psp_mapped_msa = msa.set_organisme_and_situation(df_psp_mapped_msa)

In [ ]:
# Format date_naissance to datetime python object for processing
df_psp_mapped_msa = partners.parse_beneficiary_birthdate(df_psp_mapped_msa, '%Y%m%d')

In [ ]:
# Build the qf-batch input: only ARS-origin rows carry the allocataire pivot identity
# needed for quotient_familial. One call per household, not per child, since several
# beneficiary rows can share the same allocataire. The 6-17 ans window (partners.QF_DOB_MIN /
# partners.QF_DOB_MAX) is applied here so we never spend a quotient_familial call on a
# household with no child in the window.
df_qf_allocataires, df_qf_route = partners.select_qf_route_allocataires(df_psp_mapped_msa)

# nom_usage: MSA holds it in nom_destinataire, except on a file under legal guardianship
# where the destinataire is the guardian body - set before format_qf_identity_fields, which
# leaves an existing nom_usage alone.
df_qf_allocataires = msa.build_allocataire_nom_usage(df_qf_allocataires)
df_qf_allocataires = partners.format_qf_identity_fields(df_qf_allocataires)

# code_pays_naissance: the file holds the country *label* (FRANCE, MAROC, PORTUGAL...),
# not a code - mapped here to its INSEE COG code through v_pays_territoire (France -> 99100,
# Maroc -> 99350...).
df_cog_pays = pd.read_csv(cog_pays_input_filepath, dtype=str, keep_default_na=False)
cog_by_country_label = partners.build_country_cog_lookup(df_cog_pays)

df_qf_allocataires, unmapped_labels = partners.map_birth_country_to_cog(
    df_qf_allocataires, cog_by_country_label)
if unmapped_labels:
    print(f"{len(unmapped_labels)} birth country label(s) without a COG match: {unmapped_labels}")

df_qf_allocataires, born_abroad_count = partners.clear_foreign_birthplace_insee(df_qf_allocataires)
print(f"{born_abroad_count} allocataire(s) born outside France: "
      "allocataire-code_insee_naissance cleared")

df_qf_batch_input = partners.select_qf_batch_columns(df_qf_allocataires)
df_qf_batch_input.to_csv(qf_batch_input_filepath, index=False, encoding='utf-8')

ars_rows = (df_psp_mapped_msa['situation_origine'] == 'ARS').sum()
unparsed_dob = df_qf_batch_input['allocataire-date_naissance'].isna().sum()
print(f"{len(df_qf_batch_input)} allocataire(s) (from {len(df_qf_route)} of {ars_rows} ARS row(s) "
      f"within the 6-17 ans window) written to {qf_batch_input_filepath} for qf-batch "
      f"({unparsed_dob} with an unparsed birthdate)")

## ⏸ qf-batch input is ready
The cell above wrote `MSA_QF_BATCH_INPUT_PATHFILE_2026`. qf-batch.ts can be started on it
right away (detached, can take up to a week - see worker/src/scripts/qf-batch.ts), pointing
its output at `MSA_QF_BATCH_OUTPUT_PATHFILE_2026`. The rest of this notebook only touches
beneficiary-level data, so it runs in parallel and doesn't wait for that verdict.

In [ ]:
# remove the raw columns now joined and mapped (see msa.MSA_RAW_COLUMNS_TO_DROP)
df_psp_mapped_msa = msa.drop_raw_msa_columns(df_psp_mapped_msa)

In [ ]:
# remove rows with missing necessary values (if one of those value are missing we cannot
# generate a code), then columns with all null value
df_valid = partners.filter_rows_missing_required_fields(df_psp_mapped_msa)

In [ ]:
# Upper case these columns for the merge
df_valid = partners.normalize_identity_casing(df_valid)

In [ ]:
# lower case on emails on all
df_valid = partners.normalize_email_casing(df_valid)

In [ ]:
# Preliminary filter, ahead of the precise QF/AAH/AEEH windows below: 1996-01-01 is the
# oldest birthdate any of the 3 routes can accept (AAH's lower bound, partners.AAH_DOB_MIN).
df_valid_after = partners.filter_within_eligibility_floor(df_valid)

print(f"{len(df_valid) - len(df_valid_after)} rows removed because they are outside all eligibility windows")

In [ ]:
# add missing 0 to phone numbers, and set '0' phone values to None
df_valid_after = partners.fix_phone_number_formatting(df_valid_after)

In [ ]:
# set Nan values for not existing courriel
df_valid_after = partners.clear_blank_email(df_valid_after)

In [ ]:
# add 4h on all birthdates
df_valid_after = partners.shift_birthdate_by_hours(df_valid_after)

In [ ]:
# remove duplicate beneficiaries (see partners.DEDUPLICATION_KEY_COLUMNS)
df_valid_no_duplicate, duplicate_count = partners.drop_duplicate_beneficiaries(df_valid_after)

print(f"{duplicate_count} duplicate rows were removed")

In [ ]:
# map allocataire json - MSA carries the allocataire's birth details on top of the shared
# core (see msa.ALLOCATAIRE_JSON_EXTRA_FIELDS)
df_valid_no_duplicate = partners.add_allocataire_json_column(
    df_valid_no_duplicate, msa.ALLOCATAIRE_JSON_EXTRA_FIELDS)

In [ ]:
# map adresse_allocataire json - MSA also carries nom_adresse_postale
df_valid_no_duplicate = partners.add_adresse_allocataire_json_column(
    df_valid_no_duplicate, msa.ADRESSE_JSON_EXTRA_FIELDS)

In [ ]:
# Handoff to clean_msa_2_after_qf_batch.ipynb, which resumes from here once qf-batch has
# finished. The qf-batch pivot columns are still on the frame at this point: phase 2 keys
# the verdict back onto each household with them, then drops them itself.
df_valid_no_duplicate.to_parquet(msa_intermediate_filepath)

print(f"{len(df_valid_no_duplicate)} beneficiary row(s) written to {msa_intermediate_filepath}")

## ⏭ Next: clean_msa_2_after_qf_batch.ipynb
Once qf-batch.ts has finished and written `MSA_QF_BATCH_OUTPUT_PATHFILE_2026`, open
`clean_msa_2_after_qf_batch.ipynb`: it reads the parquet above, joins the quotient familial
verdict back onto every beneficiary row of each allocataire, splits the QF/AAH/AEEH routes
and writes `DB_MSA_EXPORT_2026`.